# Task 2 — Customer Segmentation using K-Means

**Objective:** Segment an e-commerce company's customers based on purchasing behaviour so that targeted marketing strategies can be designed.

**Dataset:** UCI Online Retail (`Online Retail.xlsx`)

**Tech Stack:** Python, pandas, NumPy, scikit-learn (KMeans, StandardScaler), matplotlib, seaborn, Jupyter Notebook

### Workflow
1. Load and inspect the dataset
2. Clean missing/inconsistent transaction data
3. Create RFM features — Recency, Frequency, Monetary
4. Standardise the features
5. Use the Elbow Method to select K
6. Apply K-Means clustering
7. Visualise clusters
8. Profile each customer segment
9. Count customers per cluster
10. Recommend marketing actions


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

pd.set_option("display.max_columns", None)

df = pd.read_excel("Online Retail.xlsx")
print("Dataset shape:", df.shape)
df.head()


## 1. Dataset Inspection

The dataset contains transaction-level information such as invoice number, product code, quantity, invoice date, unit price, customer ID and country.

In [ ]:
print("Rows, Columns:", df.shape)
print("\nColumn information:")
df.info()

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

print("\nDescriptive statistics:")
df.describe(include="all").T


## 2. Data Cleaning

For customer segmentation, a customer identifier is essential, so transactions without `CustomerID` are removed.

Cancelled invoices are identified by invoice numbers beginning with `C` and excluded. Non-positive quantities and unit prices are also excluded because they do not represent normal completed purchases.

A transaction-level `TotalAmount` is calculated as:

**TotalAmount = Quantity × UnitPrice**


In [ ]:
clean = df.copy()

initial_rows = len(clean)

# Keep transactions linked to a customer
clean = clean[clean["CustomerID"].notna()].copy()

# Remove cancelled invoices
clean = clean[~clean["InvoiceNo"].astype(str).str.startswith("C")].copy()

# Keep valid purchases
clean = clean[(clean["Quantity"] > 0) & (clean["UnitPrice"] > 0)].copy()

# Calculate transaction value
clean["TotalAmount"] = clean["Quantity"] * clean["UnitPrice"]

print("Rows before cleaning:", initial_rows)
print("Rows after cleaning:", len(clean))
print("Rows removed:", initial_rows - len(clean))
print("Customers available:", clean["CustomerID"].nunique())

clean.head()


## 3. Feature Selection — RFM Analysis

We use three behavioural features:

- **Recency:** Number of days since the customer's most recent purchase. Lower is better.
- **Frequency:** Number of unique invoices/purchases made by the customer. Higher is better.
- **Monetary:** Total amount spent by the customer. Higher is better.

These three features provide a compact representation of customer purchasing behaviour.


In [ ]:
snapshot_date = clean["InvoiceDate"].max() + pd.Timedelta(days=1)

rfm = clean.groupby("CustomerID").agg(
    Recency=("InvoiceDate", lambda x: (snapshot_date - x.max()).days),
    Frequency=("InvoiceNo", "nunique"),
    Monetary=("TotalAmount", "sum")
)

print("RFM customer table shape:", rfm.shape)
rfm.describe().round(2)


In [ ]:
# RFM distribution plots
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

sns.histplot(rfm["Recency"], kde=True, ax=axes[0])
axes[0].set_title("Recency Distribution")

sns.histplot(rfm["Frequency"], kde=True, ax=axes[1])
axes[1].set_title("Frequency Distribution")

sns.histplot(rfm["Monetary"], kde=True, ax=axes[2])
axes[2].set_title("Monetary Distribution")

plt.tight_layout()
plt.show()


## 4. Normalisation / Standardisation

RFM variables have very different scales and are strongly right-skewed. We first apply `log1p` to reduce the effect of extreme values, then use `StandardScaler` so that each feature contributes on a comparable scale to K-Means.

In [ ]:
rfm_features = rfm[["Recency", "Frequency", "Monetary"]].copy()

# Reduce skewness while keeping zero values valid
rfm_log = np.log1p(rfm_features)

scaler = StandardScaler()
X = scaler.fit_transform(rfm_log)

print("Standardised feature matrix shape:", X.shape)
print("Feature means after scaling:", X.mean(axis=0).round(4))
print("Feature std after scaling:", X.std(axis=0).round(4))


## 5. Elbow Method — Choosing the Number of Clusters

We calculate K-Means inertia for several values of K. The elbow indicates a reasonable balance between model simplicity and within-cluster compactness.

In [ ]:
inertias = []
k_values = range(2, 9)

for k in k_values:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    model.fit(X)
    inertias.append(model.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(list(k_values), inertias, marker="o")
plt.xlabel("Number of clusters (K)")
plt.ylabel("Inertia")
plt.title("Elbow Method for K-Means")
plt.xticks(list(k_values))
plt.grid(alpha=0.3)
plt.show()

elbow_table = pd.DataFrame({"K": list(k_values), "Inertia": inertias})
elbow_table


### K Selection

For this dataset, **K = 4** gives a useful and interpretable segmentation. The next cell also calculates silhouette scores as an additional diagnostic.

In [ ]:
silhouette_scores = []

for k in k_values:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = model.fit_predict(X)
    silhouette_scores.append(silhouette_score(X, labels))

score_table = pd.DataFrame({
    "K": list(k_values),
    "Silhouette Score": silhouette_scores
})

plt.figure(figsize=(8, 5))
plt.plot(list(k_values), silhouette_scores, marker="o")
plt.xlabel("Number of clusters (K)")
plt.ylabel("Silhouette score")
plt.title("Silhouette Score by K")
plt.xticks(list(k_values))
plt.grid(alpha=0.3)
plt.show()

score_table


## 6. Apply K-Means Clustering

We now fit the final K-Means model with **4 clusters** and add the cluster label to the RFM table.

In [ ]:
optimal_k = 4

kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
rfm["Cluster"] = kmeans.fit_predict(X)

print("Final silhouette score:", round(silhouette_score(X, rfm["Cluster"]), 3))
rfm.head()


## 7. Cluster Visualisation

Two feature combinations are used as required:

1. Recency vs Monetary
2. Frequency vs Monetary

The plots use the original RFM values so the business meaning remains easy to understand.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.scatterplot(
    data=rfm, x="Recency", y="Monetary",
    hue="Cluster", palette="tab10", alpha=0.65, ax=axes[0]
)
axes[0].set_title("Clusters: Recency vs Monetary")

sns.scatterplot(
    data=rfm, x="Frequency", y="Monetary",
    hue="Cluster", palette="tab10", alpha=0.65, ax=axes[1]
)
axes[1].set_title("Clusters: Frequency vs Monetary")

plt.tight_layout()
plt.show()


## 8. Profile Each Cluster

Mean and median RFM values are calculated for each cluster. These values help convert mathematical clusters into meaningful customer segments.

In [ ]:
cluster_profile = rfm.groupby("Cluster").agg(
    Customers=("Cluster", "size"),
    Avg_Recency=("Recency", "mean"),
    Median_Recency=("Recency", "median"),
    Avg_Frequency=("Frequency", "mean"),
    Median_Frequency=("Frequency", "median"),
    Avg_Monetary=("Monetary", "mean"),
    Median_Monetary=("Monetary", "median")
).round(2)

cluster_profile


In [ ]:
# Assign business-friendly names based on the RFM profile
def segment_name(row):
    if row["Avg_Frequency"] >= 10 and row["Avg_Monetary"] >= 5000 and row["Avg_Recency"] <= 30:
        return "High-Value Loyal Customers"
    if row["Avg_Recency"] <= 30 and row["Avg_Frequency"] < 10:
        return "Recent / Promising Customers"
    if row["Avg_Recency"] < 120:
        return "Regular Customers"
    return "At-Risk / Inactive Customers"

name_map = {cluster: segment_name(row) for cluster, row in cluster_profile.iterrows()}
rfm["Segment"] = rfm["Cluster"].map(name_map)
cluster_profile["Segment"] = cluster_profile.index.map(name_map)

cluster_profile[[
    "Segment", "Customers", "Avg_Recency", "Avg_Frequency", "Avg_Monetary"
]].sort_index()


## 9. Number of Customers per Cluster

In [ ]:
cluster_counts = (
    rfm.groupby(["Cluster", "Segment"])
       .size()
       .reset_index(name="Customers")
       .sort_values("Cluster")
)

cluster_counts


In [ ]:
plt.figure(figsize=(10, 5))
sns.barplot(data=cluster_counts, x="Segment", y="Customers")
plt.title("Number of Customers in Each Segment")
plt.xlabel("Customer Segment")
plt.ylabel("Number of Customers")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()


## 10. Marketing Recommendations

| Customer Segment | Behaviour | Recommended Action |
|---|---|---|
| **High-Value Loyal Customers** | Very recent purchases, high frequency and high spending | VIP rewards, early access, premium bundles and loyalty benefits |
| **Recent / Promising Customers** | Recent buyers with moderate purchase frequency | Cross-sell complementary products and encourage a second/third purchase |
| **Regular Customers** | Moderate recency, frequency and spending | Personalised offers, product recommendations and loyalty points |
| **At-Risk / Inactive Customers** | Long time since purchase and low activity | Win-back emails, limited-time discounts and re-engagement campaigns |

**Business insight:** The strongest marketing priority should be retaining high-value customers while using targeted win-back campaigns for inactive customers.


## 11. Save the Customer Segmentation Dataset

The final customer-level dataset contains RFM features, cluster IDs and business-friendly segment names.

In [ ]:
rfm.to_csv("customer_segmentation_rfm.csv", index=True)

print("Saved: customer_segmentation_rfm.csv")
print("Final customer records:", len(rfm))
print("Final columns:", list(rfm.columns))


## Final Conclusion

K-Means clustering was successfully applied to customer-level RFM data. The analysis created four interpretable customer segments based on recency, purchase frequency and monetary value. These segments can be used to design targeted retention, cross-selling and re-engagement campaigns.